# Popularity Recommender (Baseline)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
users_movies = pd.read_csv("users_movies.csv")

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## Obtain the number of Ratings for each movie (build out datatable)

In [4]:
num_ratings = train.groupby("movie_id")["rating"].count()
x = num_ratings.sort_values(ascending=False)
x = x.index.tolist()
print(x[:10])

[2858, 260, 1196, 1210, 480, 589, 2028, 2571, 1270, 593]


## Obtain mean rating for each movie (build out datatable)

In [5]:
train['num_ratings'] = train['movie_id'].map(num_ratings)
filtered = train[train['num_ratings'] > 1000]
mean_ratings = filtered.groupby("movie_id")["rating"].mean()
y = mean_ratings.sort_values(ascending=False)
y = y.index.tolist()
print(y)

[318, 858, 50, 527, 750, 260, 1198, 2762, 908, 912, 1193, 1221, 2028, 593, 2858, 1136, 2571, 1197, 1196, 296, 1213, 1225, 541, 608, 919, 1247, 2804, 1617, 110, 3114, 1304, 1704, 111, 1214, 1230, 1, 1240, 1291, 2997, 1200, 2918, 2396, 1036, 356, 457, 1259, 3578, 1206, 1307, 1387, 150, 2959, 1961, 924, 589, 1610, 3471, 1210, 1394, 1270, 1079, 2791, 1097, 1265, 1220, 3481, 223, 2599, 1784, 1923, 32, 590, 2716, 1374, 1968, 2000, 3751, 34, 2355, 2797, 1073, 3408, 3793, 588, 3175, 480, 2700, 1393, 1584, 1580, 3753, 2706, 733, 1356, 3418, 2987, 2406, 2916, 1127, 3527, 1527, 380, 39, 21, 592, 1721, 2291, 377, 2174, 780, 3176, 648, 2628, 1573, 2683, 3623, 1544, 2699]


## Build both recommenders based on number of ratings, and mean rating

In [8]:
train_dict = train.groupby('user_id')['movie_id'].apply(list).to_dict()

def popularity_recommender_num(user_id, x, k=10):
    watched = list(train_dict[user_id])
    recommend_x =[]
    for i in x:
        if i not in watched:
            recommend_x.append(i)
        if len(recommend_x) == k:
            break
    return recommend_x

def popularity_recommender_mean(user_id, y, k =10):
    watched = list(train_dict[user_id])
    recommend_y =[]
    for i in y:
        if i not in watched:
            recommend_y.append(i)
        if len(recommend_y) == k:
            break
    return recommend_y
    

## Example Recommendation for User 1

In [9]:
def map_ids_to_titles(movie_ids, movies):
    movie_map = dict(zip(movies["movie_id"], movies["title"]))
    return [movie_map.get(mid, "Unknown") for mid in movie_ids]
movie_ids = popularity_recommender_num(1, x)
print("Top 10 movie recommendations for User 1\n")
print(map_ids_to_titles(movie_ids, users_movies))

Top 10 movie recommendations for User 1

['American Beauty (1999)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'Jurassic Park (1993)', 'Terminator 2: Judgment Day (1991)', 'Matrix, The (1999)', 'Silence of the Lambs, The (1991)', 'Raiders of the Lost Ark (1981)', 'Men in Black (1997)', 'Braveheart (1995)']


In [10]:
def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


## Test recommenders on all users in the Test Set

### Test the number of ratings recommender on top 10 and top 100

In [11]:
test_dict = test.groupby('user_id')['movie_id'].apply(list).to_dict()

precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0
total_hits = 0

for user_id, movies in test_dict.items():
    r = popularity_recommender_num(user_id, x, k=10)
    hits = len(set(r).intersection(movies))
    precision = hits / 10
    if len(movies) > 0: 
        recall = hits / len(movies) 
    else:
        recall = 0
    ndcg = ndcg_at_k(r, movies, k=10)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    total_hits += hits
    num_users += 1

print("Users evaluated:", num_users)
print("Total hits:", total_hits)
print("Precision@10:", precision_sum / num_users)
print("Recall@10:", recall_sum / num_users)
print("NDCG@10:", ndcg_sum / num_users)

Users evaluated: 6040
Total hits: 10987
Precision@10: 0.18190397350993698
Recall@10: 0.06825712543452876
NDCG@10: 0.20334713442800154


In [12]:
test_dict = test.groupby('user_id')['movie_id'].apply(list).to_dict()

precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0
total_hits = 0

for user_id, movies in test_dict.items():
    r = popularity_recommender_num(user_id, x, k=100)
    hits = len(set(r).intersection(movies))
    precision = hits / 100
    if len(movies) > 0: 
        recall = hits / len(movies) 
    else:
        recall = 0
    ndcg = ndcg_at_k(r, movies, k=100)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    total_hits += hits
    num_users += 1

print("Users evaluated:", num_users)
print("Total hits:", total_hits)
print("Precision@100:", precision_sum / num_users)
print("Recall@100:", recall_sum / num_users)
print("NDCG@100:", ndcg_sum / num_users)

Users evaluated: 6040
Total hits: 53144
Precision@100: 0.08798675496688545
Recall@100: 0.29592566967447886
NDCG@100: 0.2274275148456986


### Test mean recommender on top 10

In [13]:
test_dict = test.groupby('user_id')['movie_id'].apply(list).to_dict()

precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0
total_hits = 0

for user_id, movies in test_dict.items():
    r = popularity_recommender_mean(user_id, y)
    hits = len(set(r).intersection(movies))
    precision = hits / 10
    if len(movies) > 0: 
        recall = hits / len(movies) 
    else:
        recall = 0
    ndcg = ndcg_at_k(r, movies, k=10)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    total_hits += hits
    num_users += 1

print("Users evaluated:", num_users)
print("Total hits:", total_hits)
print("Precision@10:", precision_sum / num_users)
print("Recall@10:", recall_sum / num_users)
print("NDCG@10:", ndcg_sum / num_users)

Users evaluated: 6040
Total hits: 8181
Precision@10: 0.1354470198675535
Recall@10: 0.04779896225158799
NDCG@10: 0.14020631545003337
